In [29]:
import pandas as pd
import torch    
from torch.utils.data import Dataset, DataLoader
from torch import nn
import numpy as np
import matplotlib.pyplot as plt

## 1) Import libraries

We import NumPy/Pandas for data creation and PyTorch utilities for dataset and dataloader construction.

In [30]:
# define a cvs data file
path = r'data.csv'

x = np.random.rand(100, 1)
y = 2 * x + 1 
data = np.hstack((x, y))
df = pd.DataFrame(data, columns=['x', 'y'])

df.to_csv(path, index=False)


In [31]:
# define a simple dataset

class Simpleset(Dataset):
    def __init__(self, csv_path):
        self.data = pd.read_csv(csv_path)
        
        x = self.data['x'].values
        y = self.data['y'].values
        
        self.x = torch.tensor(x, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [32]:
# define a simple dataloader
dataset = Simpleset(path)

dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

for batch_x, batch_y in dataloader:
    print(batch_x.shape, batch_y.shape)
    break

torch.Size([16, 1]) torch.Size([16, 1])


In [35]:
import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info
import pandas as pd

class LargeCSVDataset(IterableDataset):
    def __init__(self, file_path, chunksize=10):
        self.file_path = file_path
        self.chunksize = chunksize

    def __iter__(self):
        worker_info = get_worker_info()
        print(f"Worker info: {worker_info}")

        if worker_info is None:
            start = 0
            step = 1
        else:
            start = worker_info.id
            step = worker_info.num_workers
            

        reader = pd.read_csv(self.file_path, chunksize=self.chunksize)

        for chunk_idx, chunk in enumerate(reader):
            if chunk_idx % step != start:
                continue

            x = chunk["x"].to_numpy(dtype="float32")
            y = chunk["y"].to_numpy(dtype="float32")

            for xi, yi in zip(x, y):
                yield xi, yi

dataset = LargeCSVDataset("data.csv", chunksize=10)

dataloader = DataLoader(
    dataset,
    batch_size=64,
    num_workers=0,
    pin_memory=True
)

for batch_x, batch_y in dataloader:
    print(batch_x.shape, batch_y.shape)
    break

Worker info: None
torch.Size([64]) torch.Size([64])


c:\Users\weijiexia\job_code_practice\MLScientistNoLeetcodeTechQuestions\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
